In [0]:
df = spark.sql("""
                SELECT 
                    price, 
                    cast(cast(`timestamp`as date) as string) as date_new
                FROM default.bronze_xauusd
            """)

In [0]:
def CRUD(df, path):
    file_path = f"/Volumes/workspace/default/invest/{path}"

    (df
    .write
    .format("delta")
    .mode("overwrite")
    .save(file_path)
    )
    
    spark.sql("use catalog workspace")
    spark.sql("use schema default") 

    if spark.catalog.tableExists(f"workspace.default.{path}"):
        spark.sql(f"""
            INSERT INTO workspace.default.{path}
            SELECT * FROM delta.`/Volumes/workspace/default/invest/{path}`
        """)
    else:
        spark.sql(f"""
            CREATE TABLE workspace.default.{path}
            AS SELECT * FROM delta.`/Volumes/workspace/default/invest/{path}`
        """)

In [0]:
try:
    API_KEY = dbutils.widgets.get("API_KEY")
except Exception as e:
    API_KEY = "S5GVQ2MWVD0UEWWU"

In [0]:
import requests

url_fx = (
    "https://www.alphavantage.co/query"
    "?function=CURRENCY_EXCHANGE_RATE"
    "&from_currency=USD"
    "&to_currency=BRL"
    f"&apikey={API_KEY}"
)
fx_data = requests.get(url_fx).json()
usd_brl = float(fx_data["Realtime Currency Exchange Rate"]["5. Exchange Rate"])

In [0]:
from pyspark.sql.functions import col, to_date

df_silver = df.withColumn("price", col("price") * usd_brl / 31.1035 * 0.75)

In [0]:
CRUD(df_silver, "silver_gold_18k")